# S05 -- daugiabazė paieška

**A kelias (be programavimo).** Kiekviena duomenų bazė grąžina atsakymą sava forma (PubMed -- XML, kitos -- JSON su skirtingais laukais). Šis demonstracinis langelis parodo neapdorotą atsakymą ir tai, kaip jis paverčiamas vienu standartiniu įrašų sąrašu. Pasirinkite šaltinius ir paleiskite 1-2 langelius.

Pastaba: čia naudojami **iš anksto išsaugoti** atsakymai, todėl užklausos tekstas rezultato nekeičia; tikra paieška reikalautų kiekvienos duomenų bazės API rakto.

**B kelias (programuotojams)** -- žemiau.

### Prieš pradedant

1. **Pirmas paleidimas.** Paleidus pirmą kodo langelį, Colab gali parodyti įspėjimą *„Notebook was not authored by Google“*. Tai normalu (sąsiuvinis atkeltas iš GitHub): spauskite **Run anyway**.
2. **Pirmas langelis** atsisiunčia kodą ir įdiegia bibliotekas (maždaug 1-2 min.). Paleiskite jį pirmą ir palaukite, kol pasirodys `Paruosta:`.
3. **Tikras DKM nebūtinas.** Jei norite jį išbandyti: kairėje Colab juostoje spauskite rakto piktogramą (*Secrets*), *Add new secret*, pavadinimas lygiai **`OPENAI_API_KEY`**, reikšmė -- jūsų OpenAI raktas, ir įjunkite *Notebook access*. Raktą rašykite tik čia, niekada į kodo langelį. Kreipiniai apmokami.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rimraul/slr-automation.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
for req in ['examples/chapter/requirements-colab.txt']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("pip install nepavyko (zr. klaida aukščiau): " + req)
print("Paruosta:", os.getcwd())

In [ ]:
#@title 1. Pasirinkite šaltinius
OPENALEX = True  #@param {type:"boolean"}
PUBMED = True  #@param {type:"boolean"}
CROSSREF = True  #@param {type:"boolean"}
SCOPUS = True  #@param {type:"boolean"}
SHOW_RAW_CHARS = 500  #@param {type:"slider", min:0, max:2000, step:100}

In [ ]:
#@title 2. Gauti, standartizuoti ir sujungti
from examples.chapter.s05_multi_db_search import solution as s5

names = [n for n, on in (("openalex", OPENALEX), ("pubmed", PUBMED),
                         ("crossref", CROSSREF), ("scopus", SCOPUS)) if on]
adapters = [s5.FixtureAdapter(n) for n in names]
if SHOW_RAW_CHARS:
    print("NEAPDOROTI ATSAKYMAI (pradžia)")
    for a in adapters:
        print(f"\n--- {a.name} ---\n{a.fetch('').strip()[:SHOW_RAW_CHARS]} ...")

records = s5.run_search(adapters, "review automation")
print(f"\nSUJUNGTA: {len(records)} įrašai iš {len(adapters)} šaltinių (dublikatai NEšalinami)\n")
print(f"{'šaltinis':9} {'metai':5} {'DOI':28} pavadinimas")
for r in records:
    print(f"{r['source']:9} {str(r['year']):5} {str(r['doi'] or '-')[:28]:28} {r['title'][:70]}")

from collections import defaultdict
by_title = defaultdict(list)
for r in records:
    by_title[" ".join(r["title"].lower().split())].append(r["source"])
same = {t: s for t, s in by_title.items() if len(s) > 1}
print(f"\nTAS PATS DARBAS keliuose šaltiniuose (tikslus pavadinimo sutapimas): {len(same)}")
for t, s in same.items():
    print(f"  {t[:70]}  <- {', '.join(s)}")
print("Panašūs, bet ne identiški pavadinimai ar įrašai be DOI lieka atskirai -- tai S6 darbas.")

---
## B kelias: parašykite kodą patys

Toliau esantis pratimas skirtas tiems, kas nori išmokti programuoti.

# S05 pratimas

```
S5 pratimas -- daugiabazė paieška.

Užduotis
--------
Kiekviena duomenų bazė grąžina savo formos atsakymą:
  * ``raw_responses/pubmed.xml``    -- PubMed EFetch XML;
  * ``raw_responses/openalex.json`` -- OpenAlex (santrauka „inverted index",
    DOI kaip URL);
  * ``raw_responses/crossref.json`` -- Crossref (pavadinimas -- sąrašas,
    santrauka su JATS žymomis);
  * ``raw_responses/scopus.json``   -- Scopus (raktai su vardų sritimis).

1. Įgyvendinkite ``normalise(name, raw_body)`` OpenAlex, PubMed ir Crossref
   šaltiniams. Kiekvienas įrašas privalo turėti TIKSLIAI raktus
   ``EXPECTED_KEYS``; ``abstract`` niekada nėra ``None`` (naudokite ``""``);
   ``doi`` -- normalizuotas (be URL prefikso) arba ``None``.
2. Įgyvendinkite ``run_search(adapters, query)`` -- sujunkite visų adapterių
   įrašus (dublikatų NEŠALINKITE).
3. SAVO užduotis: pridėkite ketvirtąjį adapterį (Scopus) savo ``exercise.py``.
4. Pastebėkite: tas pats darbas gali ateiti iš PubMed be DOI ir iš Crossref
   su DOI -- tai lieka du įrašai, kuriuos suderins S6.

Paleidimas: ``python -m examples.chapter.s05_multi_db_search.exercise``
```

1. Paleiskite paruošimo langelį. 2. Užpildykite `TODO` funkcijas pratimo langelyje ir paleiskite jį. 3. Palyginkite su pavyzdiniu sprendimu.

In [ ]:

from __future__ import annotations

from pathlib import Path

EXPECTED_KEYS = {"id", "source", "source_id", "doi", "title", "abstract", "year", "authors"}

_DATA = Path("examples/chapter") / "data" / "raw_responses"


class FixtureAdapter:
    def __init__(self, name: str) -> None:
        self.name = name
        suffix = "xml" if name == "pubmed" else "json"
        self._path = _DATA / f"{name}.{suffix}"

    def fetch(self, query: str) -> str:  # noqa: ARG002
        return self._path.read_text(encoding="utf-8").replace("\r\n", "\n")


def normalise(name: str, raw_body: str) -> list[dict]:
    """TODO: OpenAlex + PubMed + Crossref (Scopus -- jūsų užduotis)."""
    raise NotImplementedError


def run_search(adapters, query: str) -> list[dict]:
    """TODO: sujunkite visų adapterių normalizuotus įrašus."""
    raise NotImplementedError


def main() -> None:
    adapters = [FixtureAdapter(n) for n in ("openalex", "pubmed", "crossref")]
    print("S5 multi-database search exercise")
    try:
        records = run_search(adapters, "review automation")
    except NotImplementedError:
        print("normalise()/run_search() not implemented yet -- fill them in and re-run.")
        return
    print(f"merged {len(records)} records from {len(adapters)} sources")


main()

## Pavyzdinis sprendimas

Pažiūrėkite tik pabandę patys. Žemiau esantys testai tikrina **pavyzdinį sprendimą**, ne jūsų kodą.

In [ ]:
!cat examples/chapter/s05_multi_db_search/solution.py

In [ ]:
!python -m pytest examples/chapter/s05_multi_db_search -q

In [ ]:
# NEPRIVALOMA: tikri kreipiniai i DKM (sumoka uz naudojimą). Raktą laikykite
# Colab "Secrets" (rakto piktograma kairėje) pavadinimu OPENAI_API_KEY; niekada ne užrašykite kode.
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")